# Batch Normalization — лаб-тетрадь

**Статья:** Ioffe & Szegedy — *Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift* (arXiv:1502.03167, ICML 2015)

🌳 **Узел дерева:** Deep learning → optimization / normalization → BatchNorm

**Что строим:** сам BN-слой (train-time forward со статистиками по батчу + обучаемый аффин) и глубокий MLP, где BN стоит между линейным слоем и нелинейностью. Обучаем end-to-end на реальных данных.

**Что воспроизводим (эффект статьи):** при горячем LR глубокая сеть с насыщающейся нелинейностью без BN застревает, а с BN — учится быстро и добивает до высокой точности. Контраст контролируемый: одна и та же архитектура/оптимизатор/LR/сид, BN включён vs выключен.

**Данные:** sklearn `digits` (1797 картинок 8×8, 10 классов) — реальные, маленькие, CPU. Глубокий MLP на `tanh` + горячий `lr=0.6` делают стабилизирующий эффект BN наглядным.

**Компьют:** CPU, минуты.

**Parts:** (1) `batchnorm_forward` — ядро метода → (2) `Net.forward` — куда воткнуть BN в блок → (3) обучение BN vs plain + кривые (эффект статьи) → (4) ablation «пощупать фишку».

## ⛏ Эксперименты-первыми (заполни ДО чтения метода)

Открой статью, посмотри Results (Fig. 1, Fig. 2, Sec. 4.2) и ответь своими словами.

> **(a) Что результаты реально показали vs abstract?** Abstract обещает «ускорение обучения» и «можно выкинуть Dropout». Что из графиков реально видно — ускорение по шагам, финальное качество, устойчивость к LR?
> …

> **(b) Самый слабый эксперимент / чем бы фальсифицировала?** BN мешает train/test статистиками. Где авторы могли схитрить с eval-протоколом?
> …

> **(c) Где сломалось / чего НЕ протестировали?** Маленький батч? RNN? Что с «internal covariate shift» как объяснением — его вообще измерили?
> …

## Зачем это

Глубокую сеть тяжело учить: активация слоя $k$ зависит от всех слоёв до него, и по мере шагов SGD распределение входа каждого слоя ползёт. Приходится брать маленький LR и аккуратно инициализировать, иначе градиенты либо взрываются, либо насыщают нелинейность и умирают. Авторы называют этот дрейф *internal covariate shift*.

**Фишка BN.** Вместо того чтобы надеяться, что распределения сами устаканятся, BN насильно нормирует вход каждого слоя по текущему батчу — нулевое среднее, единичная дисперсия по каждому признаку — и добавляет обучаемый аффин $\gamma,\beta$, чтобы сеть при желании могла нормировку отменить. Что это даёт:

- вход нелинейности держится в её «живой» зоне → градиент не умирает в насыщении;
- лосс-ландшафт становится глаже → можно взять LR в разы больше и не разойтись;
- нормировка по батчу подмешивает шум → лёгкая регуляризация (отсюда «можно ослабить Dropout»).

Альтернатива без BN — подбирать LR и инициализацию вручную под каждую глубину. BN снимает эту зависимость: тот же горячий LR, который убивает plain-сеть, для BN-сети — просто быстрый режим.

**Почему такой датасет.** Нам нужен контраст «застрял vs учится» на CPU за минуты. `digits` маленький, а глубокий `tanh`-MLP на нём при `lr=0.6` — ровно тот режим, где plain-сеть насыщается и стоит, а BN проезжает насквозь. CIFAR тут не нужен — механизм виден и на 8×8.

## Теория (Фейнман)

**Задача.** Дан батч активаций $x \in \mathbb{R}^{B\times D}$ ($B$ примеров, $D$ признаков). Хотим подать на нелинейность что-то с предсказуемым масштабом, независимо от того, что творится в слоях ниже.

**Механизм.** Нормируем *каждый признак отдельно* по батчу:
$$\mu_j = \frac{1}{B}\sum_{i} x_{ij}, \qquad \sigma_j^2 = \frac{1}{B}\sum_{i} (x_{ij}-\mu_j)^2,$$
$$\hat{x}_{ij} = \frac{x_{ij}-\mu_j}{\sqrt{\sigma_j^2+\varepsilon}}, \qquad y_{ij} = \gamma_j\,\hat{x}_{ij} + \beta_j.$$
Среднее и дисперсия считаются *по оси батча* (по $i$), не по признакам. $\varepsilon$ — маленькая добавка, чтобы не делить на ноль. Аффин $\gamma_j,\beta_j$ обучаемый: если сети выгодно вернуть исходный масштаб, она выставит $\gamma_j=\sigma_j,\ \beta_j=\mu_j$ — то есть BN ничего не *отнимает* у выразительности.

**Что метод измеряет / даёт.** Ключевое — $\mu,\sigma$ считаются из *текущего батча* и участвуют в графе, поэтому градиент течёт и через нормировку (не только через аффин). Именно это делает шаг устойчивым: масштаб входа нелинейности зафиксирован конструкцией, а не удачей инициализации.

**Дисперсия — biased (делитель $B$, не $B-1$).** В train-time BN так и определён.

*Где у меня дыра в понимании:* …

❓ **Inline Question 1.** Если убрать обучаемый аффин ($\gamma\equiv1,\beta\equiv0$), какую выразительность сеть теряет? Приведи слой, для которого фиксированное нулевое среднее на входе нелинейности — плохо.

In [ ]:
# --- setup (готово, просто запусти) ---
# pip install torch scikit-learn matplotlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

SEED = 10          # init/shuffle seed (сид train/test-сплита зафиксирован = 0)
LR = 0.6           # горячий LR — на нём и держится весь контраст
EPOCHS = 20
HIDDEN = 128
DEPTH = 7          # столько скрытых слоёв, что plain-сеть застревает
BATCH = 64
FLOOR = 0.402      # пол для порогов (примерно уровень застрявшей plain-сети)

def load_data():
    d = load_digits()
    X = d.data.astype(np.float32) / 16.0          # 0..16 -> 0..1
    y = d.target.astype(np.int64)
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.3, random_state=0, stratify=y)
    return (torch.tensor(Xtr), torch.tensor(ytr),
            torch.tensor(Xte), torch.tensor(yte))

Xtr, ytr, Xte, yte = load_data()
print('train', Xtr.shape, 'test', Xte.shape, '| classes', int(ytr.max())+1)

## Part 1 — `batchnorm_forward` (ядро метода)

Это вклад статьи: реализуй train-time BN-forward. Вход — батч активаций $[B,D]$, обучаемые $\gamma,\beta$ формы $[D]$. Нормируй по батчу и наложи аффин. Статистики берём из текущего батча (train-режим), так что операция дифференцируема по всем трём аргументам.

Сверься с формулами из блока «Теория».

In [ ]:
def batchnorm_forward(x, gamma, beta, eps=1e-5):
    """Train-time batch normalization по оси батча.

    Каждый признак приводится к нулевому среднему и единичной дисперсии по
    текущему батчу, затем накладывается обучаемый по-признаковый аффин.
    Статистики считаются из этого же батча, поэтому результат дифференцируем
    по x, gamma и beta (градиент течёт и через нормировку).

    Args:
        x: [B, D] float — батч активаций, B примеров, D признаков.
        gamma: [D] float — обучаемый по-признаковый масштаб.
        beta:  [D] float — обучаемый по-признаковый сдвиг.
        eps: float — добавка под корень для численной устойчивости.

    Returns:
        [B, D] float — нормированные-и-аффинно-преобразованные активации;
        дифференцируемо по x, gamma, beta. До аффина каждый столбец-признак
        имеет ~0 среднее и ~1 std по батчу.

    Note: дисперсия biased (делитель B, т.е. unbiased=False), не B-1.
    """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# (1) форма и статистики: до аффина (gamma=1,beta=0) выход нормирован по батчу
torch.manual_seed(0)
x = torch.randn(256, 16) * 3.0 + 5.0     # заведомо не нормирован
g = torch.ones(16); b = torch.zeros(16)
y = batchnorm_forward(x, g, b)
assert y.shape == x.shape, f'форма поехала: {y.shape}'
m = y.mean(0); s = y.var(0, unbiased=False)
print(f'Expected \u2248 mean 0, var 1 | у тебя mean|max|={m.abs().max():.2e}, var\u2208[{s.min():.3f},{s.max():.3f}]')
assert m.abs().max() < 1e-4, 'среднее по БАТЧУ (dim=0) не занулилось — по той ли оси считаешь?'
assert (s - 1).abs().max() < 1e-2, 'дисперсия по батчу не 1 — biased (unbiased=False)? делишь на sqrt(var+eps)?'

# (2) аффин: gamma/beta масштабируют и сдвигают ровно по-признаково
g2 = torch.arange(1, 17).float(); b2 = torch.full((16,), 2.0)
y2 = batchnorm_forward(x, g2, b2)
assert torch.allclose(y2.mean(0), b2, atol=1e-3), 'beta должен задавать новое среднее по каждому признаку'
assert torch.allclose(y2.std(0, unbiased=False), g2, atol=1e-2), 'gamma должен задавать новый std по каждому признаку'

# (3) grad-flow: градиент течёт и в x, и в gamma, и в beta
xg = x.clone().requires_grad_(True); gg = g.clone().requires_grad_(True); bg = b.clone().requires_grad_(True)
batchnorm_forward(xg, gg, bg).pow(2).sum().backward()
assert xg.grad is not None and gg.grad is not None and bg.grad is not None, 'нет градиента по одному из аргументов'
assert xg.grad.abs().sum() > 0, 'градиент по x нулевой — нормировка выпала из графа (detach? .data?)'
print('ok')

❓ **Inline Question 2.** В тесте (1) `x` имеет std=3 и среднее=5, а выход — std≈1, mean≈0. Через backward это значит, что $\partial y/\partial x$ *масштабируется* на $\approx 1/\sigma$. Как это связано с тем, что BN «чинит» градиент в глубокой сети?

**Ваш ответ:** …

## Part 2 — куда воткнуть BN в блоке

Теперь собери глубокий MLP. Каркас класса дан (слои, параметры $\gamma,\beta$, голова уже созданы в `__init__`). Твоя задача — `forward`: провести вход через `DEPTH` скрытых блоков и голову. В каждом скрытом блоке участвуют линейный слой `self.linears[i]`, нелинейность `tanh` и — если `self.use_bn` — твой `batchnorm_forward` с `self.gammas[i]`, `self.betas[i]`.

Порядок трёх операций внутри блока — часть метода, и он не произвольный: цель BN — чтобы `tanh` не уезжал в насыщение (см. Sec. 3.1 статьи). Тест `gamma=0` ниже проверит, что ты собрал блок так, что BN реально влияет на вход нелинейности.

In [ ]:
class Net(nn.Module):
    """Глубокий MLP, опционально с BatchNorm в каждом скрытом блоке.

    При use_bn=False — обычный глубокий tanh-MLP (наша plain-бейзлайн).
    """
    def __init__(self, in_dim, n_classes, hidden=HIDDEN, depth=DEPTH, use_bn=True):
        super().__init__()
        self.use_bn = use_bn
        dims = [in_dim] + [hidden] * depth
        self.linears = nn.ModuleList(
            [nn.Linear(dims[i], dims[i + 1]) for i in range(depth)])
        if use_bn:
            self.gammas = nn.ParameterList(
                [nn.Parameter(torch.ones(hidden)) for _ in range(depth)])
            self.betas = nn.ParameterList(
                [nn.Parameter(torch.zeros(hidden)) for _ in range(depth)])
        self.head = nn.Linear(hidden, n_classes)

    def forward(self, x):
        """Прогон батча через глубокий MLP.

        Каждый скрытый блок сочетает линейный слой self.linears[i], нелинейность
        tanh и — при self.use_bn — batchnorm_forward с self.gammas[i]/self.betas[i].
        После всех скрытых блоков — линейная голова self.head.

        Args:
            x: [B, in_dim] float — входной батч.

        Returns:
            [B, n_classes] float — логиты классов (до softmax).
        """
        # YOUR CODE HERE
        raise NotImplementedError

In [ ]:
torch.manual_seed(0)
net = Net(64, 10, use_bn=True)
out = net(Xtr[:32])
assert out.shape == (32, 10), f'на выходе должны быть логиты [B,10], а не {tuple(out.shape)}'
assert torch.isfinite(out).all(), 'NaN/inf на forward — порядок lin/bn/tanh?'

# BN реально в графе: если занулить все gamma, tanh(0)=0 -> голова видит константу ->
# все строки логитов одинаковые (с точностью до общего bias головы)
with torch.no_grad():
    for gpar in net.gammas: gpar.zero_()
flat = net(Xtr[:32])
assert (flat - flat[0:1]).abs().max() < 1e-4, 'gamma=0 не схлопнул выход — BN не применяется к входу tanh?'

# grad-flow до самого нижнего линейного слоя
net2 = Net(64, 10, use_bn=True)
F.cross_entropy(net2(Xtr[:32]), ytr[:32]).backward()
g0 = net2.linears[0].weight.grad
assert g0 is not None and g0.abs().sum() > 0, 'нет градиента в первом слое — цепочка блоков разорвана'
print('ok')

❓ **Inline Question 3.** В `__init__` при `use_bn=True` заводится по паре $\gamma,\beta$ на каждый скрытый слой, но *не* на выход головы. Почему BN обычно не ставят прямо перед softmax-логитами?

**Ваш ответ:** …

## Part 3 — обучение: BN vs plain (эффект статьи)

Тренировочная обвязка дана (SGD, горячий `LR=0.6`, один и тот же сид для обеих сетей). Запусти обе и посмотри на кривые тестовой точности по эпохам.

Это контролируемый контраст: всё одинаково, кроме `use_bn`. Ждём, что plain-сеть застрянет, а BN-сеть быстро доедет до высокой точности.

In [ ]:
def accuracy(net, X, y):
    net.eval()
    with torch.no_grad():
        pred = net(X).argmax(1)
    return (pred == y).float().mean().item()

def train(use_bn, seed=SEED):
    torch.manual_seed(seed); np.random.seed(seed)
    net = Net(Xtr.shape[1], 10, use_bn=use_bn)
    opt = torch.optim.SGD(net.parameters(), lr=LR)
    n = Xtr.shape[0]; curve = []
    for ep in range(EPOCHS):
        net.train(); perm = torch.randperm(n)
        for i in range(0, n, BATCH):
            idx = perm[i:i + BATCH]
            opt.zero_grad()
            loss = F.cross_entropy(net(Xtr[idx]), ytr[idx])
            loss.backward(); opt.step()
        curve.append(accuracy(net, Xte, yte))
    return net, curve

def first_epoch_reaching(curve, thresh=0.90):
    for ep, a in enumerate(curve):
        if a >= thresh:
            return ep
    return None

net_bn, curve_bn = train(use_bn=True)
net_pl, curve_pl = train(use_bn=False)
acc_bn, acc_plain = curve_bn[-1], curve_pl[-1]
best_bn = max(curve_bn)
ep_bn, ep_pl = first_epoch_reaching(curve_bn), first_epoch_reaching(curve_pl)
print(f'acc_bn    final={acc_bn:.3f}  best={best_bn:.3f}  90%@epoch={ep_bn}')
print(f'acc_plain final={acc_plain:.3f}                90%@epoch={ep_pl}')

In [ ]:
# Пороги захардкожены и НЕ подгоняются — калиброваны по verify-прогону reference.
# Держатся на зашитом сиде и подтверждены на 5 доп. сидах.
print(f'Expected: acc_bn>{FLOOR+0.3:.3f}; acc_bn>acc_plain; BN 90%@\u2264ep5; BN не позже plain')
print(f'         acc_bn={acc_bn:.3f}  acc_plain={acc_plain:.3f}  ep_bn={ep_bn}  ep_pl={ep_pl}')
assert acc_bn > FLOOR + 0.3, f'BN не оторвался от пола: {acc_bn:.3f} <= {FLOOR+0.3:.3f}'
assert acc_bn > acc_plain, f'BN не лучше plain: {acc_bn:.3f} <= {acc_plain:.3f}'
assert ep_bn is not None and ep_bn <= 5, f'BN не взял 90% к эпохе 5: {ep_bn}'
assert ep_pl is None or ep_bn <= ep_pl, 'BN взял 90% позже plain'
print('ok — эффект статьи воспроизведён')

## Ключевая фигура — кривые обучения

Headline статьи в наших числах: скорость выхода на плато и разрыв между BN и plain.

In [ ]:
import matplotlib.pyplot as plt
ep = range(1, EPOCHS + 1)
plt.figure(figsize=(7, 4))
plt.plot(ep, curve_bn, marker='o', label=f'BN (best {max(curve_bn):.3f})')
plt.plot(ep, curve_pl, marker='s', label=f'plain (final {curve_pl[-1]:.3f})')
plt.axhline(0.9, ls='--', c='gray', lw=1, label='90%')
plt.xlabel('epoch'); plt.ylabel('test accuracy')
plt.title(f'BatchNorm vs plain · deep tanh-MLP (depth {DEPTH}) · lr={LR}')
plt.legend(); plt.grid(True, alpha=0.3); plt.ylim(0, 1); plt.show()

Обрати внимание: у BN-кривой *final* нередко ниже *best* — на горячем LR траектория на крошечном датасете подпрыгивает, финальная эпоха ловит просадку. Направление и величина эффекта от этого не меняются (best BN ≫ plain при любой сводной статистике), но если сравнивать сети «по финалу», держи бунс в голове.

❓ **Inline Question 4.** Plain-сеть с таким LR фактически стоит на месте с первой эпохи. Что именно с ней происходит на forward — посмотри, в каком диапазоне активации `tanh` в её слоях (насыщение → $\tanh'\approx0$ → мёртвый градиент). Как это связано с тем, что BN держит вход `tanh` нормированным?

**Ваш ответ:** …

## Part 4 — ablation: пощупать фишку (устойчивость к LR)

Главное обещание BN — не «чуть лучше», а «можно взять LR, на котором plain разваливается». Проверим напрямую: прогоним обе сети по сетке LR и посмотрим, у кого лучшая тестовая точность держится, а у кого проваливается.

Обвязка свипа дана. Смотри, где кривые расходятся.

In [ ]:
lrs = [0.03, 0.1, 0.3, 0.6, 1.0]
bn_by_lr, pl_by_lr = [], []
for lr in lrs:
    _LR = LR
    globals()['LR'] = lr
    _, cb = train(use_bn=True)
    _, cp = train(use_bn=False)
    globals()['LR'] = _LR
    bn_by_lr.append(max(cb)); pl_by_lr.append(max(cp))
    print(f'lr={lr:<4}  BN best={max(cb):.3f}   plain best={max(cp):.3f}')

plt.figure(figsize=(7, 4))
plt.plot(lrs, bn_by_lr, marker='o', label='BN')
plt.plot(lrs, pl_by_lr, marker='s', label='plain')
plt.xscale('log'); plt.xlabel('learning rate'); plt.ylabel('best test acc')
plt.title('Устойчивость к LR: BN vs plain'); plt.legend(); plt.grid(True, alpha=0.3)
plt.ylim(0, 1); plt.show()

Вот она, фишка: на малом LR обе сети более-менее живы (plain медленнее, но едет), а по мере разогрева LR plain-кривая проваливается — насыщается и стоит, — тогда как BN держит качество. То есть BN не столько «поднимает потолок», сколько *расширяет диапазон LR*, где обучение вообще идёт.

❓ **Inline Question 5.** В какой новой ситуации ты потянешься за BatchNorm (или его роднёй — LayerNorm/GroupNorm) в первую очередь? Сформулируй как «если я вижу симптом X, ставлю нормировку». И: почему для трансформеров/RNN обычно берут LayerNorm, а не BatchNorm — что ломается в батч-статистиках?

**Ваш ответ:** …

## Debug-area (если что-то не сходится)

- **`batchnorm_forward` даёт NaN.** Скорее всего `sqrt` от отрицательного или деление на ~0 — проверь `+eps` под корнем и что дисперсия считается по `dim=0`.
- **Тест (1) валит `mean`.** Считаешь среднее по признакам (`dim=1`), а надо по батчу (`dim=0`, `keepdim=True`).
- **Тест (1) валит `var`.** Взял `unbiased=True` (делитель B−1). BN требует biased.
- **`Net.forward` даёт одинаковые логиты уже без зануления gamma.** Скорее всего забыл `tanh` или прогоняешь не через все слои.
- **grad по первому слою нулевой.** Где-то в цепочке `.detach()`/`.data`, или BN применяется к копии, а не к тензору в графе.
- **BN-кривая скачет как бешеная.** Это ожидаемо на `lr=0.6` — смотри на `best`, не только на `final`. Свип LR (Part 4) показывает, что при меньшем LR траектория глаже.

## Рефлексия

**Что забрать в работу:**
> Нормировка активаций — дешёвый способ развязать обучение от инициализации и открыть горячий LR. …

**В какой новой ситуации BN/нормировка — правильный инструмент:**
> Симптом: глубокая сеть, обучение сверхчувствительно к LR/инициализации, активации насыщаются или разъезжаются по масштабу. …

**Что осталось непонятным → следующий узел beam:**
> Train/eval-расхождение (running stats vs batch stats), маленькие батчи, почему LayerNorm вытеснил BN в трансформерах, «действительно ли дело в internal covariate shift» (Santurkar et al. 2018 — smoothing лосс-ландшафта). …

## SPOILERS — открывай, только если застряла

Сначала попробуй сама; NaN и мутные кривые здесь — норма, а не признак провала.

<details><summary>Part 1 — batchnorm_forward</summary>

```python
mean = x.mean(dim=0, keepdim=True)
var = x.var(dim=0, unbiased=False, keepdim=True)
xhat = (x - mean) / torch.sqrt(var + eps)
return gamma * xhat + beta
```
Ключ: статистики по `dim=0` (ось батча), `unbiased=False`, `+eps` под корнем. `gamma`/`beta` формы `[D]` бродкастятся по батчу.
</details>

<details><summary>Part 2 — Net.forward</summary>

```python
for i, lin in enumerate(self.linears):
    x = lin(x)
    if self.use_bn:
        x = batchnorm_forward(x, self.gammas[i], self.betas[i])
    x = torch.tanh(x)
return self.head(x)
```
BN идёт *после* линейного и *до* `tanh` — так нормированный вход попадает в живую зону нелинейности. Голова — просто `self.head(x)`, без BN и без активации.
</details>